# Task 3 · Independent re-judging with an open model on Colab

**Why.** In our Task 3 run the same model (Claude) wrote the ideas *and* judged them. A judge can prefer its own
style, so we ask a different model family to re-judge **exactly the same judge prompts** and measure how often the
two judges agree. `rejudge.py` does the comparison; this notebook gives it a judge without rate limits by running an
open model (default **Qwen3-8B**) on the Colab GPU with vLLM.

**What it re-judges** (437 calls, about 2.5M prompt tokens):
- Stage 1: the 60 problem reviews (1–5 per criterion) → Spearman correlation, exact and within-1 agreement
- Stage 2: HypER relevance scores (0/1/2) of 167 citing papers → weighted kappa
- Stage 3: the 336 pairwise comparisons → winner agreement and a Bradley–Terry refit
- Stage 4: the 24 idea reviews → Spearman per criterion and the top 5 recomputed from the new reviews

**Runtime:** Runtime → Change runtime type → **A100 GPU** (L4 also works with Qwen3-8B). About 30–45 minutes in total,
most of it installing vLLM and downloading the model.

Nothing here changes the Task 3 results; it adds `outputs/rejudge_<model>.json`, which the documentation PDF reports.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the Task 3 code

Option A clones our branch (works if the repository is public, or with a GitHub token in the URL).
Option B: upload `task3_run3.zip` to the Colab file panel (or to Google Drive) and unzip it instead.

In [ ]:
import os, subprocess
REPO = "https://github.com/Divyam10113/Intro-to-LLMs.git"   # for a private repo: https://<token>@github.com/Divyam10113/Intro-to-LLMs.git
BRANCH = "task3_kevalina"
ZIP = "/content/task3_run3.zip"                              # used if cloning fails

if not os.path.exists("/content/repo/task3"):
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "/content/repo"], capture_output=True, text=True)
    if r.returncode != 0:
        print("clone failed, trying the zip:", r.stderr[-300:])
        os.makedirs("/content/repo", exist_ok=True)
        subprocess.run(["unzip", "-q", ZIP, "-d", "/content/repo"], check=True)
%cd /content/repo/task3
!ls

## 2. Install vLLM and the pipeline's dependencies (5–10 min)

In [ ]:
!pip install -q vllm openai scikit-learn scipy pandas
!python -c "import vllm; print('vLLM', vllm.__version__)"

## 3. Choose the judge model and start the server

- `Qwen/Qwen3-8B` (default): fits A100 and L4, 32k context (our longest prompt is about 27k tokens).
- `Qwen/Qwen3-14B`: a stronger judge, A100 only.
- Any other open model works if vLLM supports it and its context is at least 32k.

`THINKING = False` asks Qwen3 to answer directly (faster, closer to how the prompts were answered). With
`THINKING = True` the model reasons first; its reasoning is separated from the answer by vLLM's reasoning parser.

In [ ]:
MODEL = "Qwen/Qwen3-8B"
THINKING = False
MAX_LEN = 32768
CONCURRENCY = 16

import subprocess, time, urllib.request, json
cmd = ["vllm", "serve", MODEL, "--max-model-len", str(MAX_LEN), "--gpu-memory-utilization", "0.90", "--port", "8000"]
if "qwen3" in MODEL.lower():
    cmd += ["--reasoning-parser", "qwen3"]
log = open("/content/vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
for i in range(180):                      # wait up to 30 min for the download and start-up
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        print(f"server ready after {i * 10} s"); break
    except Exception:
        if server.poll() is not None:
            print(open("/content/vllm.log").read()[-3000:]); raise SystemExit("vLLM stopped; see the log above")
        time.sleep(10)
else:
    print(open("/content/vllm.log").read()[-3000:])

In [ ]:
from openai import OpenAI
cli = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
extra = {"chat_template_kwargs": {"enable_thinking": THINKING}} if "qwen3" in MODEL.lower() else None
r = cli.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Reply with the word ready."}],
                                max_tokens=200, extra_body=extra)
print(r.choices[0].message.content)

## 4. Re-judge

The settings go to `llm_client.py` through environment variables. Every answer is cached in `logs/cache/` under the
new model's name, so if the session disconnects, re-running this cell continues where it stopped.

In [ ]:
import os, json
os.environ.update(JUDGE_BACKEND="openai", JUDGE_BASE_URL="http://localhost:8000/v1", JUDGE_API_KEY="none",
                  JUDGE_MODEL=MODEL, JUDGE_CONCURRENCY=str(CONCURRENCY), JUDGE_MAX_TOKENS="6000" if THINKING else "1500")
if extra:
    os.environ["JUDGE_EXTRA_BODY"] = json.dumps(extra)
!python3 rejudge.py 2>&1 | tail -n 60

## 5. Read the result

In [ ]:
import glob, json, re
f = sorted(glob.glob("outputs/rejudge_*.json"), key=os.path.getmtime)[-1]
R = json.load(open(f)); print(f)
print(f"Stage 1 problem reviews: Spearman {R['stage1']['spearman']}, exact {R['stage1']['exact']}, within 1 point {R['stage1']['within1']} (n={R['stage1']['n']})")
print(f"Stage 2 relevance 0/1/2: exact {R['stage2']['exact']}, weighted kappa {R['stage2']['qwk']} (n={R['stage2']['n']})")
for pid, v in R["stage3"].items():
    print(f"Stage 3 {pid}: same winner in {v['winner_agreement']:.0%} of {v['n']} comparisons; fitness ranking Spearman {v['bt_spearman']}; best idea: ours {v['original_best']}, new judge {v['new_best']}")
for pid, v in R["stage4"].items():
    print(f"Stage 4 {pid}: review-mean Spearman {v['spearman_mean']}; top-5 overlap {v['top5_overlap']}/5; new top 5 {v['top5_new']}")

**How to read it.** Agreement between two different LLM judges is usually moderate: HypER reports Cohen's kappa
0.46 between its teacher model and experts, and ResearchAgent's human–model agreement is of a similar order. High
overlap of the top-5 lists matters more than criterion-level correlations, because the top 5 is what Task 4 uses.

## 6. Save the results

Download the zip (or copy it to Drive) and unzip it **inside your local `task3/` folder**: it adds
`outputs/rejudge_<model>.json` and the new judge's cached answers. Then run `python3 make_reports.py` to add the
re-judging section to the documentation, and commit.

In [ ]:
import zipfile, glob, os, json
tag = re.sub(r"[^A-Za-z0-9.-]+", "_", MODEL)
z = f"/content/rejudge_{tag}.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob(f"outputs/rejudge_{tag}.json"):
        zf.write(f)
    n = 0
    for f in glob.glob("logs/cache/*.json"):
        if json.load(open(f)).get("tag", "").startswith("rejudge/"):
            zf.write(f); n += 1
print(z, "with", n, "cached answers")
try:
    from google.colab import files; files.download(z)
except Exception as e:
    print("download from the file panel:", e)

In [ ]:
server.terminate()   # free the GPU when done